# SQLite Store
> fastlite-based hybrid SQLite session store

In [ ]:
#| default_exp store.sqlite

In [ ]:
#| export
import orjson
from pathlib import Path
from typing import Optional, List
from fastlite import database
from memexcode.core import gen_id, MessageEntry, CompactionEntry, SessionHeader
from memexcode.session import SessionEntry
from memexcode.pluginspec import SessionABC, SessionStore, SessionInfo, TreeNode, hookimpl

In [ ]:
#| export
class SqliteSession(SessionABC):
    def __init__(self, db, session_id: str):
        self.db = db
        self._session_id = session_id

    async def append(self, entry: SessionEntry) -> None:
        if not entry.id: entry._entry.id = gen_id()
        entries = self.db.t.entries
        last = entries.rows_where('session_id = ? ORDER BY rowid DESC LIMIT 1', (self._session_id,),
                                  select='id').fetchone()
        entry._entry.parentId = last['id'] if last else None
        entries.insert(dict(
            id=entry.id, session_id=self._session_id, parent_id=entry.parentId,
            type=entry.type, timestamp=entry.timestamp, data=orjson.dumps(entry.to_dict())
        ))

    async def get_by_id(self, entry_id: str) -> Optional[SessionEntry]:
        row = self.db.t.entries.rows_where('id = ? AND session_id = ?', (entry_id, self._session_id)).fetchone()
        return SessionEntry.from_dict(orjson.loads(row['data'])) if row else None

    async def get_path_to_root(self, leaf_id: str) -> List[SessionEntry]:
        sql = '''
        WITH RECURSIVE path(id, parent_id, type, timestamp, data, depth) AS (
            SELECT id, parent_id, type, timestamp, data, 0
            FROM entries WHERE id = :leaf AND session_id = :sid
            UNION ALL
            SELECT e.id, e.parent_id, e.type, e.timestamp, e.data, p.depth + 1
            FROM entries e JOIN path p ON e.id = p.parent_id
            WHERE e.session_id = :sid
        )
        SELECT * FROM path ORDER BY depth DESC
        '''
        rows = self.db.execute(sql, {'leaf': leaf_id, 'sid': self._session_id}).fetchall()
        return [SessionEntry.from_dict(orjson.loads(r['data'])) for r in rows]

    async def load_all(self) -> List[SessionEntry]:
        rows = self.db.t.entries.rows_where('session_id = ? ORDER BY rowid', (self._session_id,))
        return [SessionEntry.from_dict(orjson.loads(r['data'])) for r in rows]

    async def compact(self, first_kept_id: str, summary: str) -> None:
        first = self.db.t.entries.rows_where(
            'id = ? AND session_id = ?', (first_kept_id, self._session_id)
        ).fetchone()
        if not first:
            raise ValueError(f'first_kept_id {first_kept_id} not found')
        compact = SessionEntry(CompactionEntry(
            parentId=first['parent_id'], summary=summary, firstKeptEntryId=first_kept_id
        ))
        self.db.t.entries.delete_where('rowid < (SELECT rowid FROM entries WHERE id = ? AND session_id = ?)',
                                         (first_kept_id, self._session_id))
        self.db.t.entries.insert(dict(
            id=compact.id, session_id=self._session_id, parent_id=compact.parentId,
            type=compact.type, timestamp=compact.timestamp, data=orjson.dumps(compact.to_dict())
        ))

    async def get_tree(self, root_id: Optional[str] = None) -> TreeNode:
        rows = self.db.t.entries.rows_where('session_id = ? ORDER BY rowid', (self._session_id,))
        entries = {r['id']: SessionEntry.from_dict(orjson.loads(r['data'])) for r in rows}
        children = {r['id']: [] for r in rows}
        root = None
        for r in rows:
            if not root: root = entries[r['id']]
            if r['parent_id'] and r['parent_id'] in children: children[r['parent_id']].append(entries[r['id']])
        if not root: return TreeNode(SessionEntry(MessageEntry()), [])
        def build(entry): return TreeNode(entry, [build(entries[c]) for c in children.get(entry.id, [])])
        return build(root)

class SqliteStore(SessionStore):
    def __init__(self, db_path: Path):
        self.db = database(str(db_path))
        self._ensure_schema()

    def _ensure_schema(self):
        if not self.db.t.sessions.exists():
            self.db.t.sessions.create(
                dict(id=str, cwd=str, name=str, parent_session=str, created_at=str, updated_at=str),
                pk='id'
            )
        if not self.db.t.entries.exists():
            self.db.t.entries.create(
                dict(id=str, session_id=str, parent_id=str, type=str, timestamp=str, data=bytes),
                pk='id'
            )
            self.db.t.entries.create_index(['session_id'])
            self.db.t.entries.create_index(['parent_id'])

    def get_session(self, session_id: str) -> SessionABC:
        return SqliteSession(self.db, session_id)

    async def create(self, cwd: str, name: Optional[str] = None, parent_session: Optional[str] = None) -> SessionInfo:
        info = SessionInfo(id=gen_id(), cwd=cwd, created_at=now_iso(), updated_at=now_iso(),
                          name=name, parent_session=parent_session)
        self.db.t.sessions.insert(dict(
            id=info.id, cwd=info.cwd, name=info.name,
            parent_session=info.parent_session,
            created_at=info.created_at, updated_at=info.updated_at
        ))
        return info

    async def list(self) -> List[SessionInfo]:
        rows = self.db.t.sessions.rows(order_by='created_at DESC')
        return [SessionInfo(
            id=r['id'], cwd=r['cwd'], name=r['name'],
            parent_session=r['parent_session'],
            created_at=r['created_at'], updated_at=r['updated_at']
        ) for r in rows]

    async def rename(self, session_id: str, name: str) -> None:
        self.db.t.sessions.update(session_id, dict(name=name, updated_at=now_iso()))

    async def delete(self, session_id: str) -> None:
        self.db.t.entries.delete_where('session_id = ?', (session_id,))
        self.db.t.sessions.delete(session_id)

    async def fork(self, session_id: str, leaf_id: Optional[str] = None, name: Optional[str] = None) -> SessionInfo:
        old = self.get_session(session_id)
        path = await old.get_path_to_root(leaf_id) if leaf_id else await old.load_all()
        info = await self.create(cwd='', name=name, parent_session=session_id)
        new = self.get_session(info.id)
        for entry in path: await new.append(entry)
        return info

@hookimpl
def get_store():
    return ('sqlite', SqliteStore)